<a href="https://colab.research.google.com/github/tramthanh1025/Try/blob/main/InitialEDA_Nerissa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Initial EDA on the original training set (~87M notifications)

## Mount & Extract

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = "/content/drive/MyDrive/Datathon"
WORK = "/content/work"

import os
os.makedirs(WORK, exist_ok=True)

# Extract only if empty
import glob
import subprocess

tar_files = glob.glob(f"{PROJECT}/train-part-*.tar.gz")

for f in tar_files:
    print("Extracting:", f)
    subprocess.run(["tar", "-xzf", f, "-C", WORK])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting: /content/drive/MyDrive/Datathon/train-part-3.tar.gz
Extracting: /content/drive/MyDrive/Datathon/train-part-2.tar.gz
Extracting: /content/drive/MyDrive/Datathon/train-part-1.tar.gz


In [ ]:
!ls {WORK}

train-part-1  train-part-2  train-part-3


tar.gz contains parquet files. As per the readme "The files for each part should be unzipped to a single direcctory and read as a single dataset."

## Connect DuckDB

In [ ]:
!pip install -q duckdb

In [ ]:

import duckdb

con = duckdb.connect()

train_path = "/content/work/**/*.parquet"

con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{train_path}')
""").fetchone()

(87665839,)

In [ ]:


con.execute(f"""
CREATE OR REPLACE VIEW train AS
SELECT * FROM read_parquet('{train_path}');
""")

# Initial EDA


In [ ]:
con.execute("DESCRIBE train").df()

In [ ]:
con.execute("SELECT * FROM train LIMIT 15").df()

**Data Exploration: Global Reward Rate**

In [ ]:
baseline_stats = con.execute("""
    SELECT
        COUNT(*) AS total_notifications,
        SUM(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS positive_samples,
        SUM(CASE WHEN NOT session_end_completed THEN 1 ELSE 0 END) AS negative_samples,
        AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS global_reward_rate
    FROM train
""").df()

# Extract values for easier use
total = baseline_stats['total_notifications'][0]
pos = baseline_stats['positive_samples'][0]
neg = baseline_stats['negative_samples'][0]
rate = baseline_stats['global_reward_rate'][0]

print(f"Total Notifications: {total:,}")
print(f"Positive Interactions (Success): {pos:,}")
print(f"Negative Interactions (Ignored): {neg:,}")
print(f"Global Reward Rate: {rate:.2%}")

Total Notifications: 87,665,839
Positive Interactions (Success): 12,608,610.0
Negative Interactions (Ignored): 75,057,229.0
Global Reward Rate: 14.38%


=> Result: The global reward rate on the whole training set (about 87M notifications) is 14.38%. The success ones are around 12M (14.4%) while the ignored results account for 85.6% (about 75M)

**Data Exploration: UI Language and Eligible Templates sets**

In [ ]:
distinct_languages_df = con.execute("SELECT DISTINCT ui_language FROM train ORDER BY ui_language").df()
distinct_languages = distinct_languages_df['ui_language'].tolist() # Prepare for further analysis later

print(f"Number of UI Languages: {len(distinct_languages)}")
print(f"Distinct UI Languages: {distinct_languages}")

Number of UI Languages: 25
Distinct UI Languages: ['ar', 'bn', 'cs', 'de', 'dn', 'el', 'en', 'es', 'fr', 'hi', 'hu', 'id', 'it', 'ja', 'ko', 'pl', 'pt', 'ro', 'ru', 'ta', 'th', 'tr', 'uk', 'vi', 'zs']


=> Result: 25 distinct languages.

In [ ]:
distinct_eligible_templates_df = con.execute("SELECT DISTINCT eligible_templates FROM train ORDER BY eligible_templates").df()
distinct_eligible_templates = distinct_eligible_templates_df['eligible_templates'].tolist() # Prepare for further analysis later

print(f"Number of Eligible Templates: {len(distinct_eligible_templates)}")
print(f"Distinct Eligible Templates set:")
print(distinct_eligible_templates_df)

=> Result: 9 distinct eligible templates in which only one class of users eligible to receive template C and 5 classes of users eligible to us template A.

In [ ]:
template_performance_df = con.execute("""
    SELECT
        selected_template,
        COUNT(*) AS notification_count,
        AVG(CASE WHEN session_end_completed THEN 1.0 ELSE 0.0 END) AS success_rate
    FROM train
    GROUP BY selected_template
    ORDER BY notification_count DESC
""").df()

print(template_performance_df)


In [ ]:
sorted_by_success_rate_df = template_performance_df.sort_values(by='success_rate', ascending=False)
print("Templates sorted by success rate (top 5):")
print(sorted_by_success_rate_df.head())

print("\nTemplates sorted by success rate (bottom 5):")
print(sorted_by_success_rate_df.tail())

Templates sorted by success rate (top 5):
   selected_template  notification_count  success_rate
10                 C             2523858      0.412298
9                  A             3472696      0.268958
3                  L             9062239      0.131680
6                  G             9058068      0.131586
0                  K             9129646      0.131572

Templates sorted by success rate (bottom 5):
  selected_template  notification_count  success_rate
2                 J             9062686      0.130362
4                 E             9060589      0.129533
7                 F             9057400      0.128914
1                 H             9124283      0.128724
5                 B             9059625      0.128534


=> Result:
*   Template Frequencies: Templates K, H, J, L, and E were the most frequently selected, each appearing in approximately 9 to 9.1 million notifications.
*   Success Rates of Frequent Templates: The highly frequent templates (K, H, J, L, E) exhibited relatively consistent success rates, ranging from 12.8% to 13.2%. This was noted to be slightly below the global reward rate of 14.38%.
*   Top-Performing Templates by Success Rate:
    *   Template C showed a significantly higher success rate at 41.23%, despite having 2,523,858 notifications.
    *   Template A also demonstrated a strong success rate of 26.90%, with 3,472,696 notifications.
*   Bottom-Performing Templates by Success Rate: Templates B, H, F, E, and J were among the lowest performers, with success rates ranging from 12.85% to 13.04%. Notably, some of these (H, E, J) were also among the most frequently used.


**Data Exploration: Time zone and Hours of the day**:



In [ ]:
timezone_mappings = {
    'UTC+1 (Western/Central Europe)': (1, ['de', 'fr', 'it', 'es', 'nl', 'pl', 'cs', 'el', 'hu', 'dn']),
    'UTC+0 (UK/Western Europe)': (0, ['en', 'pt']),
    'UTC+2 (Eastern Europe/Middle East)': (2, ['ru', 'uk', 'ro', 'ar', 'tr']),
    'UTC+5.5 (India)': (5, ['hi', 'bn', 'ta']),
    'UTC+7 (Southeast Asia)': (7, ['id', 'th', 'vi']),
    'UTC+9 (East Asia)': (9, ['ja', 'ko']),
    'Unknown/Other': (0, ['zs']) # 'zs' is less common code, assigned to UTC+0 as a default.
}

# Ensure all distinct_languages are covered
all_mapped_languages = set()
for offset, languages in timezone_mappings.values():
    all_mapped_languages.update(languages)

# Check for any unmapped languages from the distinct_languages list
# distinct_languages is from the kernel state, which is: ['ar', 'bn', 'cs', 'de', 'dn', 'el', 'en', 'es', 'fr', 'hi', 'hu', 'id', 'it', 'ja', 'ko', 'pl', 'pt', 'ro', 'ru', 'ta', 'th', 'tr', 'uk', 'vi', 'zs']

unmapped_languages = [lang for lang in distinct_languages if lang not in all_mapped_languages]

# Add any remaining unmapped languages to the 'Unknown/Other' category if they exist
if unmapped_languages:
    print(f"Adding unmapped languages to 'Unknown/Other': {unmapped_languages}")
    # Convert the tuple to a list to modify, then convert back to tuple
    unknown_offset, unknown_langs = timezone_mappings['Unknown/Other']
    timezone_mappings['Unknown/Other'] = (unknown_offset, sorted(list(set(unknown_langs + unmapped_languages))))


print("Timezone Mappings Dictionary created:")
for tz_name, (offset, langs) in timezone_mappings.items():
    print(f"  {tz_name}: Offset={offset}, Languages={langs}")

Timezone Mappings Dictionary created:
  UTC+1 (Western/Central Europe): Offset=1, Languages=['de', 'fr', 'it', 'es', 'nl', 'pl', 'cs', 'el', 'hu', 'dn']
  UTC+0 (UK/Western Europe): Offset=0, Languages=['en', 'pt']
  UTC+2 (Eastern Europe/Middle East): Offset=2, Languages=['ru', 'uk', 'ro', 'ar', 'tr']
  UTC+5.5 (India): Offset=5, Languages=['hi', 'bn', 'ta']
  UTC+7 (Southeast Asia): Offset=7, Languages=['id', 'th', 'vi']
  UTC+9 (East Asia): Offset=9, Languages=['ja', 'ko']
  Unknown/Other: Offset=0, Languages=['zs']


In [ ]:
hourly_reward_rates_by_timezone = {}

for tz_name, (offset, languages) in timezone_mappings.items():
    # Convert list of languages to a comma-separated string for SQL IN clause
    languages_str = ", ".join(f"'{lang}'" for lang in languages)

    # Construct the SQL query
    query = f"""
        SELECT
            CAST(MOD(datetime * 24 + {offset}, 24) AS INTEGER) AS local_hour,
            COUNT(*) AS n_notifications,
            AVG(CASE WHEN session_end_completed THEN 1.0 ELSE 0.0 END) AS reward_rate
        FROM train
        WHERE ui_language IN ({languages_str})
        GROUP BY local_hour
        ORDER BY local_hour
    """

    print(f"Processing timezone: {tz_name} (Languages: {languages})")
    # Execute the query and store the result
    hourly_reward_rates_by_timezone[tz_name] = con.execute(query).df()
    print(f"  - Data for {tz_name} loaded. {len(hourly_reward_rates_by_timezone[tz_name])} rows.\n")

print("Hourly reward rates calculated for all timezones.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate the number of subplots needed based on the number of timezones
num_timezones = len(hourly_reward_rates_by_timezone)
num_cols = 2  # Number of columns for subplots
num_rows = int(np.ceil(num_timezones / num_cols))

plt.figure(figsize=(15, 6 * num_rows))

for i, (tz_name, df) in enumerate(hourly_reward_rates_by_timezone.items()):
    plt.subplot(num_rows, num_cols, i + 1)
    plt.plot(df['local_hour'], df['reward_rate'], marker='o', linestyle='-')
    plt.title(f'Hourly Reward Rate for {tz_name}')
    plt.xlabel('Local Hour of Day (0-23)')
    plt.ylabel('Average Reward Rate')
    plt.xticks(np.arange(0, 24, 2)) # Show every other hour for clarity
    plt.grid(True)
    plt.ylim(0, df['reward_rate'].max() * 1.2) # Set y-limit dynamically based on data

plt.tight_layout()
plt.show()


=> Result:

*   Global Average vs. Timezone-Specific Rates: While the global reward rate is around 14.38%, the hourly reward rates within specific timezones show significant fluctuation, indicating that the time of day plays a crucial role in user engagement.

*   Peak Engagement Hours: Most timezone groups exhibit a clear peak in reward rates during certain local hours. This suggests that users are more receptive to notifications during these times. For example:
    *   **UTC+0 (UK/Western Europe)** and **UTC+1 (Western/Central Europe)** tend to show higher engagement in the morning (e.g., 8-10 AM) and potentially in the early evening (e.g., 5-7 PM), with a dip in the late-night/early-morning hours.
    *   **UTC+5.5 (India)** and **UTC+7 (Southeast Asia)** show patterns shifted according to their local time, with peaks typically during their daytime working/leisure hours.
    *   **UTC+9 (East Asia)** shows similar patterns to European timezones but shifted to reflect their local day, peaking during active hours.

*   Differences in Reward Rate Magnitudes: Some timezones consistently show higher average reward rates than others, even at their peak times. This could be due to cultural differences, device usage patterns, or the effectiveness of content localized for those regions.

*   Lowest Engagement Hours: Across all timezones, the lowest reward rates are generally observed during late-night and early-morning local hours, when most users are likely asleep or less active on their devices. The dip in engagement around 3-5 AM local time is consistent across many groups.

*   "Unknown/Other" Timezone: The "Unknown/Other" group, which stands for 'zs', shows a flatter reward rate curve compared to other well-defined timezones, with less pronounced peaks and troughs. This could be due to a heterogeneous mix of languages or a smaller sample size that smooths out typical daily patterns.

=> Business Implications:
These insights can be crucial for optimizing notification delivery times. Instead of sending notifications uniformly, personalizing delivery based on the user's inferred timezone and local optimal engagement hours could significantly increase the session completion rate. For instance, sending notifications to Western/Central European users in their morning hours and Indian users during their local afternoon could lead to higher success rates compared to a global, untargeted approach.